# 2. Feature Engineering & Preprocessing

This notebook transforms raw transactions into model-ready features.

**Block A: Data Cleaning** — duplicates, remaining_lease parsing, storey parsing, Y/N conversion
**Block B: Geospatial Features** — geocoding, MRT/school/mall/hawker/CBD/carpark distances
**Block C: Final Assembly** — merge building info, mature estate flag, train-test split

Output: `data/processed/train.csv` and `data/processed/test.csv`

In [44]:
%pip install geopy aiohttp nest-asyncio pyproj geopandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [45]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import os
from geopy.distance import geodesic
from src.preprocessing import (
    parse_remaining_lease, parse_storey_range,
    convert_yn_to_bool, extract_transaction_date
)
from src.feature_engineering import (
    geocode_buildings, geocode_schools,
    compute_nearest_mrt, compute_nearest_from_locations,
    compute_cbd_distance, compute_nearest_school_by_level,
    classify_elite_school
)
from src.data_collection import MATURE_ESTATES

In [46]:
df = pd.read_csv("../data/raw/resale_transactions.csv", low_memory=False)
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (692719, 11)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN
3,2000-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN
4,2000-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN


## Block A: Data Cleaning & Preprocessing

In [47]:
# Drop exact duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} exact duplicates. Shape: {df.shape}")

Dropped 1101 exact duplicates. Shape: (691618, 11)


In [48]:
# Extract transaction year/month
df = extract_transaction_date(df)

# Parse remaining_lease -> months (handles null/numeric/"X years Y months")
df["remaining_lease_months"] = parse_remaining_lease(
    df["remaining_lease"], df["month"], df["lease_commence_date"]
)
print(f"remaining_lease_months nulls: {df['remaining_lease_months'].isna().sum()}")
print(f"Range: {df['remaining_lease_months'].min():.0f} to {df['remaining_lease_months'].max():.0f} months")

# Parse storey_range -> numeric median
df["storey_median"] = parse_storey_range(df["storey_range"])
print(f"\nstorey_median nulls: {df['storey_median'].isna().sum()}")
print(f"Range: {df['storey_median'].min():.1f} to {df['storey_median'].max():.1f}")

# Drop collinear / replaced columns
df = df.drop(columns=["remaining_lease", "storey_range", "lease_commence_date"])
print(f"\nDropped remaining_lease, storey_range, lease_commence_date")
print(f"Shape: {df.shape}")
df.head()

remaining_lease_months nulls: 0
Range: 476 to 1215 months

storey_median nulls: 0
Range: 2.0 to 50.0

Dropped remaining_lease, storey_range, lease_commence_date
Shape: (691618, 12)


,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,resale_price,transaction_year,transaction_month,remaining_lease_months,storey_median
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,69.0,Improved,147000.0,2000,1,1031.0,8.0
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,61.0,Improved,144000.0,2000,1,1031.0,5.0
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,73.0,New Generation,159000.0,2000,1,911.0,8.0
3,2000-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,73.0,New Generation,167000.0,2000,1,911.0,8.0
4,2000-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,67.0,New Generation,163000.0,2000,1,911.0,8.0


## Block B: Geospatial Feature Engineering

### B1: Geocode Buildings
OneMap API to get lat/lng for each unique (block, street_name) pair. Results cached to `data/interim/building_geocodes.csv`.

In [49]:
unique_buildings = df[["block", "street_name"]].drop_duplicates().reset_index(drop=True)
print(f"Unique buildings to geocode: {len(unique_buildings)}")

building_coords = geocode_buildings(
    unique_buildings,
    cache_path="../data/interim/building_geocodes.csv",
    concurrency=10,
    batch_size=500
)

# Merge coordinates into main dataframe
df = df.merge(building_coords, on=["block", "street_name"], how="left")
print(f"\nBuildings with coordinates: {df['latitude'].notna().sum()} / {len(df)}")
print(f"Missing: {df['latitude'].isna().sum()}")

Unique buildings to geocode: 9919
Loaded 9919 cached buildings (9813 with coords)
106 buildings still need geocoding
  Batch 0-106: 0/106 successful

Total: 9919 buildings, 9813 geocoded, 106 failed


/Users/anhminhhoang/Documents/GitHub/HousePricePrediction/notebooks/../src/feature_engineering.py:89: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([cached, checkpoint], ignore_index=True).drop_duplicates(
/Users/anhminhhoang/Documents/GitHub/HousePricePrediction/notebooks/../src/feature_engineering.py:98: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([cached, new_data], ignore_index=True).drop_duplicates(



Buildings with coordinates: 687828 / 691618
Missing: 3790


### B2: Nearest MRT (Time-Varying)
For each transaction, only use MRT stations that were open at the transaction date.

In [50]:
mrt_df = pd.read_csv("../data/raw/mrt_lrt_stations.csv")
mrt_df["opening_date"] = pd.to_datetime(mrt_df["opening_date"])
print(f"MRT/LRT stations: {len(mrt_df)}")

MRT_CACHE = "../data/interim/mrt_distances.csv"
df["month_dt"] = pd.to_datetime(df["month"])

if os.path.exists(MRT_CACHE):
    mrt_cached = pd.read_csv(MRT_CACHE)
    mrt_cached["month_dt"] = pd.to_datetime(mrt_cached["month_dt"])
    print(f"Loaded {len(mrt_cached)} cached MRT distances from {MRT_CACHE}")

    df = df.merge(
        mrt_cached[["block", "street_name", "month_dt", "dist_to_nearest_mrt_km", "closest_mrt"]],
        on=["block", "street_name", "month_dt"], how="left"
    )
    filled = df["dist_to_nearest_mrt_km"].notna().sum()
    print(f"Matched {filled} / {len(df)} rows from cache")

    # Compute any missing (new buildings or new months not in cache)
    missing_mask = df["dist_to_nearest_mrt_km"].isna() & df["latitude"].notna()
    if missing_mask.sum() > 0:
        print(f"Computing {missing_mask.sum()} uncached rows...")
        missing_bm = df.loc[missing_mask, ["block", "street_name", "latitude", "longitude", "month_dt"]].drop_duplicates(
            subset=["block", "street_name", "month_dt"]
        ).reset_index(drop=True)

        new_results = []
        for i, row in missing_bm.iterrows():
            dist, name = compute_nearest_mrt(row["latitude"], row["longitude"], mrt_df, row["month_dt"])
            new_results.append({"block": row["block"], "street_name": row["street_name"],
                                "month_dt": row["month_dt"], "dist_to_nearest_mrt_km": dist, "closest_mrt": name})

        new_df = pd.DataFrame(new_results)
        # Update cache
        mrt_cached = pd.concat([mrt_cached, new_df], ignore_index=True).drop_duplicates(
            subset=["block", "street_name", "month_dt"], keep="last"
        )
        mrt_cached.to_csv(MRT_CACHE, index=False)
        print(f"Updated cache: {len(mrt_cached)} total entries")

        # Fill missing in df
        for _, row in new_df.iterrows():
            mask = (df["block"] == row["block"]) & (df["street_name"] == row["street_name"]) & (df["month_dt"] == row["month_dt"])
            df.loc[mask, "dist_to_nearest_mrt_km"] = row["dist_to_nearest_mrt_km"]
            df.loc[mask, "closest_mrt"] = row["closest_mrt"]
else:
    # First run — compute everything
    building_month = df[["block", "street_name", "latitude", "longitude", "month_dt"]].drop_duplicates(
        subset=["block", "street_name", "month_dt"]
    ).reset_index(drop=True)

    print(f"Unique (building, month) pairs: {len(building_month)}")
    print("Computing time-varying MRT distances... (this will take ~30 minutes)")

    results = []
    for i, row in building_month.iterrows():
        if pd.isna(row["latitude"]):
            results.append((np.nan, None))
            continue
        dist, name = compute_nearest_mrt(row["latitude"], row["longitude"], mrt_df, row["month_dt"])
        results.append((dist, name))
        if (i + 1) % 10000 == 0:
            print(f"  Processed {i + 1}/{len(building_month)}")

    building_month["dist_to_nearest_mrt_km"] = [r[0] for r in results]
    building_month["closest_mrt"] = [r[1] for r in results]

    # Save cache
    building_month[["block", "street_name", "month_dt", "dist_to_nearest_mrt_km", "closest_mrt"]].to_csv(
        MRT_CACHE, index=False
    )
    print(f"Saved MRT distance cache to {MRT_CACHE}")

    df = df.merge(
        building_month[["block", "street_name", "month_dt", "dist_to_nearest_mrt_km", "closest_mrt"]],
        on=["block", "street_name", "month_dt"], how="left"
    )

print(f"\nMRT distance range: {df['dist_to_nearest_mrt_km'].min():.3f} to {df['dist_to_nearest_mrt_km'].max():.3f} km")
print(f"Nulls: {df['dist_to_nearest_mrt_km'].isna().sum()}")

MRT/LRT stations: 190
Loaded 560083 cached MRT distances from ../data/interim/mrt_distances.csv
Matched 687828 / 691618 rows from cache

MRT distance range: 0.021 to 5.878 km
Nulls: 3790


### B3: Schools — Geocode, Nearest by Level, Elite Flags

In [51]:
SCHOOL_FLAGS_CACHE = "../data/interim/school_flags.csv"

schools_raw = pd.read_csv("../data/raw/schools.csv")
schools_geo = geocode_schools(schools_raw, cache_path="../data/interim/schools_geocoded.csv")
print(f"Schools with coordinates: {schools_geo['lat'].notna().sum()} / {len(schools_geo)}")

buildings_with_coords = building_coords[building_coords["latitude"].notna()].reset_index(drop=True)

if os.path.exists(SCHOOL_FLAGS_CACHE):
    school_flags = pd.read_csv(SCHOOL_FLAGS_CACHE)
    print(f"Loaded {len(school_flags)} cached school flags")
else:
    print(f"Computing nearest schools for {len(buildings_with_coords)} buildings...")
    school_results = []
    for i, row in buildings_with_coords.iterrows():
        pri, sec, mixed = compute_nearest_school_by_level(row["latitude"], row["longitude"], schools_geo)
        school_results.append({"block": row["block"], "street_name": row["street_name"],
                               "closest_pri_sch": pri, "closest_sec_sch": sec, "closest_mixed_sch": mixed})
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i + 1}/{len(buildings_with_coords)}")

    school_df = pd.DataFrame(school_results)
    school_df["is_elite_closest_pri_sch"] = school_df["closest_pri_sch"].apply(
        lambda x: classify_elite_school(x, schools_geo))
    school_df["is_elite_closest_sec_sch"] = school_df["closest_sec_sch"].apply(
        lambda x: classify_elite_school(x, schools_geo))
    school_df["is_elite_closest_mixed_sch"] = school_df["closest_mixed_sch"].apply(
        lambda x: classify_elite_school(x, schools_geo))

    school_flags = school_df[["block", "street_name", "is_elite_closest_pri_sch",
                               "is_elite_closest_sec_sch", "is_elite_closest_mixed_sch"]]
    school_flags.to_csv(SCHOOL_FLAGS_CACHE, index=False)
    print(f"Saved school flags cache to {SCHOOL_FLAGS_CACHE}")

df = df.merge(school_flags, on=["block", "street_name"], how="left")
print(f"Elite school flags added. Shape: {df.shape}")

Loaded 337 cached schools (337 with coords)
Schools with coordinates: 337 / 337
Loaded 9813 cached school flags
Elite school flags added. Shape: (691618, 20)


### B4: CBD, Mall, Hawker Distances
These are static per building (no time-varying component).

In [52]:
SPATIAL_CACHE = "../data/interim/spatial_distances.csv"

if os.path.exists(SPATIAL_CACHE):
    spatial_df = pd.read_csv(SPATIAL_CACHE)
    print(f"Loaded {len(spatial_df)} cached spatial distances")
else:
    malls_df = pd.read_csv("../data/raw/shopping_malls.csv")
    hawker_df = pd.read_csv("../data/raw/hawker_centres.csv")

    print(f"Malls: {len(malls_df)}, Hawker centres: {len(hawker_df)}")
    print(f"Computing CBD/mall/hawker distances for {len(buildings_with_coords)} buildings...")

    spatial_results = []
    for i, row in buildings_with_coords.iterrows():
        lat, lng = row["latitude"], row["longitude"]
        cbd_dist = compute_cbd_distance(lat, lng)
        mall_dist, _ = compute_nearest_from_locations(lat, lng, malls_df, "lat", "lng")
        hawker_dist, _ = compute_nearest_from_locations(lat, lng, hawker_df, "lat", "lng")
        spatial_results.append({
            "block": row["block"], "street_name": row["street_name"],
            "dist_to_cbd_km": cbd_dist,
            "dist_to_nearest_mall_km": mall_dist,
            "dist_to_nearest_hawker_km": hawker_dist,
        })
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i + 1}/{len(buildings_with_coords)}")

    spatial_df = pd.DataFrame(spatial_results)
    spatial_df.to_csv(SPATIAL_CACHE, index=False)
    print(f"Saved spatial distances cache to {SPATIAL_CACHE}")

df = df.merge(spatial_df, on=["block", "street_name"], how="left")
print(f"CBD/mall/hawker distances added. Shape: {df.shape}")

Loaded 9813 cached spatial distances
CBD/mall/hawker distances added. Shape: (691618, 23)


### B5: Car Park Features
Match HDB car parks to buildings by proximity. Convert SVY21 → WGS84 coordinates.

In [53]:
CARPARK_CACHE = "../data/interim/carpark_features.csv"

if os.path.exists(CARPARK_CACHE):
    carpark_feat_df = pd.read_csv(CARPARK_CACHE)
    print(f"Loaded {len(carpark_feat_df)} cached carpark features")
else:
    from pyproj import Transformer

    carpark_df = pd.read_csv("../data/raw/hdb_carpark_info.csv")
    transformer = Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True)

    carpark_df["x_coord"] = pd.to_numeric(carpark_df["x_coord"], errors="coerce")
    carpark_df["y_coord"] = pd.to_numeric(carpark_df["y_coord"], errors="coerce")
    valid_coords = carpark_df["x_coord"].notna() & carpark_df["y_coord"].notna()

    lngs, lats = transformer.transform(
        carpark_df.loc[valid_coords, "x_coord"].values,
        carpark_df.loc[valid_coords, "y_coord"].values
    )
    carpark_df.loc[valid_coords, "lat"] = lats
    carpark_df.loc[valid_coords, "lng"] = lngs

    print(f"Car parks: {len(carpark_df)}")
    print(f"Matching car parks to {len(buildings_with_coords)} buildings...")

    carpark_results = []
    for i, row in buildings_with_coords.iterrows():
        dist, cp_idx = compute_nearest_from_locations(
            row["latitude"], row["longitude"],
            carpark_df[carpark_df["lat"].notna()], "lat", "lng"
        )
        if cp_idx is not None:
            cp = carpark_df.loc[cp_idx]
            carpark_results.append({
                "block": row["block"], "street_name": row["street_name"],
                "nearest_carpark_dist_km": dist,
                "nearest_carpark_type": cp["car_park_type"],
            })
        else:
            carpark_results.append({
                "block": row["block"], "street_name": row["street_name"],
                "nearest_carpark_dist_km": np.nan,
                "nearest_carpark_type": None,
            })
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i + 1}/{len(buildings_with_coords)}")

    carpark_feat_df = pd.DataFrame(carpark_results)
    carpark_feat_df.to_csv(CARPARK_CACHE, index=False)
    print(f"Saved carpark features cache to {CARPARK_CACHE}")

df = df.merge(carpark_feat_df, on=["block", "street_name"], how="left")
print(f"Car park features added. Shape: {df.shape}")

Loaded 9813 cached carpark features
Car park features added. Shape: (691618, 25)


### B6: Macroeconomic Features
SORA 3M (interest rate) and CPI (inflation), merged with 1-month lag to prevent leakage.

In [54]:
# Toggle this to enable/disable macro features
ENABLE_MACRO_FEATURES = True  # ENABLED — adding SORA 3M + CPI

if ENABLE_MACRO_FEATURES:
    sora_df = pd.read_csv("../data/raw/sora_3m.csv")
    cpi_df = pd.read_csv("../data/raw/cpi.csv")

    sora_df["lag_year"] = sora_df["year"]
    sora_df["lag_month"] = sora_df["month"] + 1
    mask = sora_df["lag_month"] > 12
    sora_df.loc[mask, "lag_month"] = 1
    sora_df.loc[mask, "lag_year"] = sora_df.loc[mask, "lag_year"] + 1

    cpi_df["lag_year"] = cpi_df["year"]
    cpi_df["lag_month"] = cpi_df["month"] + 1
    mask = cpi_df["lag_month"] > 12
    cpi_df.loc[mask, "lag_month"] = 1
    cpi_df.loc[mask, "lag_year"] = cpi_df.loc[mask, "lag_year"] + 1

    df = df.merge(
        sora_df[["lag_year", "lag_month", "sora_3m"]].rename(
            columns={"lag_year": "transaction_year", "lag_month": "transaction_month", "sora_3m": "sora_3m_lagged"}
        ),
        on=["transaction_year", "transaction_month"], how="left"
    )
    df = df.merge(
        cpi_df[["lag_year", "lag_month", "cpi"]].rename(
            columns={"lag_year": "transaction_year", "lag_month": "transaction_month", "cpi": "cpi_lagged"}
        ),
        on=["transaction_year", "transaction_month"], how="left"
    )

    # Fill edge nulls: Jan 2000 SORA (no Dec 1999 lag) and recent CPI (not yet published)
    # Use nearest available value — safe for edge months
    df["sora_3m_lagged"] = df["sora_3m_lagged"].fillna(method="bfill").fillna(method="ffill")
    df["cpi_lagged"] = df["cpi_lagged"].fillna(method="ffill").fillna(method="bfill")

    print(f"SORA nulls after fill: {df['sora_3m_lagged'].isna().sum()}")
    print(f"CPI nulls after fill: {df['cpi_lagged'].isna().sum()}")
    print(f"SORA range: {df['sora_3m_lagged'].min():.2f}% to {df['sora_3m_lagged'].max():.2f}%")
    print(f"CPI range: {df['cpi_lagged'].min():.1f} to {df['cpi_lagged'].max():.1f}")
    print(f"Shape after macro merge: {df.shape}")
else:
    print("Macro features DISABLED — set ENABLE_MACRO_FEATURES = True to add SORA + CPI")

SORA nulls after fill: 0
CPI nulls after fill: 0
SORA range: 0.05% to 3.70%
CPI range: 63.2 to 102.4
Shape after macro merge: (691618, 27)


/var/folders/kz/x1bqw4g54k9bxv6zd2qlrp500000gn/T/ipykernel_38280/2001504166.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["sora_3m_lagged"] = df["sora_3m_lagged"].fillna(method="bfill").fillna(method="ffill")
/var/folders/kz/x1bqw4g54k9bxv6zd2qlrp500000gn/T/ipykernel_38280/2001504166.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["cpi_lagged"] = df["cpi_lagged"].fillna(method="ffill").fillna(method="bfill")


## Block C: Final Assembly

### C1: Merge HDB Building Property Info

In [55]:
property_df = pd.read_csv("../data/raw/hdb_property_info.csv")

# Drop time-varying columns (unit counts sold/rented) to prevent leakage
time_varying_cols = [c for c in property_df.columns if
                     "sold" in c or "rental" in c]
property_df = property_df.drop(columns=time_varying_cols)

# Rename to match transaction columns
property_df = property_df.rename(columns={"blk_no": "block", "street": "street_name"})

# Convert Y/N to boolean
yn_cols = ["residential", "commercial", "market_hawker", "miscellaneous",
           "multistorey_carpark", "precinct_pavilion"]
property_df = convert_yn_to_bool(property_df, yn_cols)

print(f"Property info: {property_df.shape}")
print(f"Columns: {list(property_df.columns)}")

df = df.merge(property_df, on=["block", "street_name"], how="left")
print(f"\nAfter merge: {df.shape}")
print(f"Unmatched: {df['max_floor_lvl'].isna().sum()} rows")

Property info: (13289, 12)
Columns: ['block', 'street_name', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units']

After merge: (691618, 37)
Unmatched: 5464 rows


### C2: Mature Estate Flag & Final Cleanup

In [56]:
# Mature estate flag
df["is_mature_estate"] = df["town"].str.upper().isin(MATURE_ESTATES)
print(f"Mature: {df['is_mature_estate'].sum()}, Non-mature: {(~df['is_mature_estate']).sum()}")

# Drop rows with missing geospatial or property features
before = len(df)
df = df.dropna(subset=["latitude", "max_floor_lvl"])
dropped = before - len(df)
print(f"\nDropped {dropped} rows with missing geo/property data ({dropped/before*100:.1f}%)")

# Drop high-cardinality columns now captured by lat/lng and other features
df = df.drop(columns=["block", "street_name", "month", "month_dt", "bldg_contract_town"], errors="ignore")

print(f"\nFinal shape: {df.shape}")
print(f"\nNull counts:")
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("None — all clean!")
else:
    print(null_counts[null_counts > 0])
print(f"\nColumns: {list(df.columns)}")
df.head()

Mature: 318452, Non-mature: 373166

Dropped 5464 rows with missing geo/property data (0.8%)

Final shape: (686154, 33)

Null counts:
None — all clean!

Columns: ['town', 'flat_type', 'floor_area_sqm', 'flat_model', 'resale_price', 'transaction_year', 'transaction_month', 'remaining_lease_months', 'storey_median', 'latitude', 'longitude', 'dist_to_nearest_mrt_km', 'closest_mrt', 'is_elite_closest_pri_sch', 'is_elite_closest_sec_sch', 'is_elite_closest_mixed_sch', 'dist_to_cbd_km', 'dist_to_nearest_mall_km', 'dist_to_nearest_hawker_km', 'nearest_carpark_dist_km', 'nearest_carpark_type', 'sora_3m_lagged', 'cpi_lagged', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'total_dwelling_units', 'is_mature_estate']


,town,flat_type,floor_area_sqm,flat_model,resale_price,transaction_year,transaction_month,remaining_lease_months,storey_median,latitude,...,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,precinct_pavilion,total_dwelling_units,is_mature_estate
0,ANG MO KIO,3 ROOM,69.0,Improved,147000.0,2000,1,1031.0,8.0,1.374001,...,10.0,1980.0,True,True,False,True,False,False,179.0,True
1,ANG MO KIO,3 ROOM,61.0,Improved,144000.0,2000,1,1031.0,5.0,1.375097,...,11.0,1980.0,True,False,False,False,False,False,198.0,True
2,ANG MO KIO,3 ROOM,73.0,New Generation,159000.0,2000,1,911.0,8.0,1.366197,...,10.0,1975.0,True,False,False,False,False,False,144.0,True
3,ANG MO KIO,3 ROOM,73.0,New Generation,167000.0,2000,1,911.0,8.0,1.366558,...,10.0,1975.0,True,True,False,False,False,False,130.0,True
4,ANG MO KIO,3 ROOM,67.0,New Generation,163000.0,2000,1,911.0,8.0,1.365119,...,13.0,1976.0,True,True,False,True,False,False,190.0,True


### C3: Train-Test Split by Date

In [57]:
# Temporal split: train before 2025-09, test from 2025-09 onwards
SPLIT_YEAR = 2025
SPLIT_MONTH = 9

train_mask = (df["transaction_year"] < SPLIT_YEAR) | (
    (df["transaction_year"] == SPLIT_YEAR) & (df["transaction_month"] < SPLIT_MONTH)
)

df_train = df[train_mask].reset_index(drop=True)
df_test = df[~train_mask].reset_index(drop=True)

print(f"Train: {len(df_train)} rows ({df_train['transaction_year'].min()}-{df_train['transaction_year'].max()})")
print(f"Test:  {len(df_test)} rows ({df_test['transaction_year'].min()}-{df_test['transaction_year'].max()})")
print(f"Test ratio: {len(df_test) / len(df) * 100:.1f}%")

# Save
os.makedirs("../data/processed", exist_ok=True)
df_train.to_csv("../data/processed/train.csv", index=False)
df_test.to_csv("../data/processed/test.csv", index=False)
print(f"\nSaved to ../data/processed/train.csv and test.csv")

Train: 667327 rows (2000-2025)
Test:  18827 rows (2025-2026)
Test ratio: 2.7%

Saved to ../data/processed/train.csv and test.csv


In [58]:
# Final summary
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"\nTotal features: {len(df.columns) - 1} (excluding resale_price)")
print(f"\nFeature list:")
for col in sorted(df.columns):
    dtype = df[col].dtype
    nulls = df[col].isna().sum()
    print(f"  {col:35s} {str(dtype):10s} nulls={nulls}")
print(f"\nTrain: {len(df_train)} | Test: {len(df_test)}")

FEATURE ENGINEERING COMPLETE

Total features: 32 (excluding resale_price)

Feature list:
  closest_mrt                         object     nulls=0
  commercial                          object     nulls=0
  cpi_lagged                          float64    nulls=0
  dist_to_cbd_km                      float64    nulls=0
  dist_to_nearest_hawker_km           float64    nulls=0
  dist_to_nearest_mall_km             float64    nulls=0
  dist_to_nearest_mrt_km              float64    nulls=0
  flat_model                          object     nulls=0
  flat_type                           object     nulls=0
  floor_area_sqm                      float64    nulls=0
  is_elite_closest_mixed_sch          object     nulls=0
  is_elite_closest_pri_sch            object     nulls=0
  is_elite_closest_sec_sch            object     nulls=0
  is_mature_estate                    bool       nulls=0
  latitude                            float64    nulls=0
  longitude                           float64    nulls=0